# Fine Tuning
Fine-tuning in machine learning refers to the process of taking a pre-trained model and further training it on a smaller, task-specific dataset to adapt it to a new but related problem.

How it works:
1. Start with a model pre-trained on a large, general dataset (e.g., ImageNet, or a language model like GPT).
2. Freeze some or all of the original model's layers (optional).
3. Train (or re-train) the remaining layers using the new dataset, typically with a lower learning rate.
4. The model learns task-specific patterns while retaining general knowledge from the pre-training

We will follow an [example](https://huggingface.co/docs/transformers/en/training) from huggingface for fine tuning.

We will do the following:
1. Fine tuning the model against the training data `yelp_review_full`, where inputs are reviews, outputs are stars from 1 - 5.
2. Use the `google-bert/bert-base-cased` tokenizer for tokenization
3. Use the `google-bert/bert-base-cased` as the base model for fine tuning

In [ ]:
%pip install transformers==4.51.3 datasets==3.5.1 evaluate==0.4.3 numpy==2.2.4

In [18]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
dataset = load_dataset("yelp_review_full")

In [19]:
dataset['train'][0]

{'label': 4,
 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."}

The original data size is large, for learning purpose, we limit the size of the data to only 1000 rows.

In [20]:
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval = dataset["test"].shuffle(seed=42).select(range(1000))

In [28]:
def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

small_train = small_train.map(tokenize, batched=True)
small_eval = small_eval.map(tokenize, batched=True)

# Set the format to "torch" to have the features directly accessible
small_train.set_format("torch", columns=['input_ids', 'attention_mask', 'label'])
small_eval.set_format("torch", columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [29]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Prepare to fine tune the model, we will use the following params:
- `eval_strategy="epoch"`: evaluate the loss after each epoch
- `save_strategy="no"`: we do not save the trained model anywhere, ideally we should save it on disk so that we can reload it
- `report_to="none"`: set to "none" to not integrate with any weights tracking platform (such as "wandb")

In [31]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.119715,0.527000
2,No log,1.013838,0.566000
3,No log,1.034243,0.592000


TrainOutput(global_step=375, training_loss=1.03717578125, metrics={'train_runtime': 357.9414, 'train_samples_per_second': 8.381, 'train_steps_per_second': 1.048, 'total_flos': 789354427392000.0, 'train_loss': 1.03717578125, 'epoch': 3.0})

In [36]:
predictions = trainer.predict(small_eval)
predicted_labels = predictions.predictions.argmax(-1)  # if classification

In [50]:
index = 1
example = small_eval[index]
input = tokenizer.decode(example["input_ids"], skip_special_tokens=True)
expected = example["label"]
predicted = predicted_labels[index]
print("Input:", input)
print("Expected:", expected)
print("Predicted:", predicted)

Input: Visiting here for 10 days and staying in a condo \ nDriving around and found this place \ nAte here 2x ( Lunch + Dinner ) \ nFood was fresh + Tasty \ nSushi was great for the price \ nShrimp OK but about the same as other buffets \ nSesame balls were lite and great \ nFor desert they had a coffee taramisu which was rather good
Expected: tensor(4)
Predicted: 3


Now that we have finished fine tuning the model, we can observe from the training log that our accuracy is around 59.2, which is fine, but not great enough to make accurate prediections.
Given that the whole training process was around 6 minutes, this is accepetable. For better performance we will need to:
1. Train on the full size data instead of the small one
2. Train with more epochs